# Customer churn: model development and evaluation

A synthetic-data portfolio project using pandas and scikit-learn. The objective is to predict next-month churn and choose a model and decision threshold under illustrative error costs.

**Scope:** This notebook generates synthetic data, explores development data, compares logistic regression with a constrained random forest, evaluates the selection procedure, and exports a fitted model. PostgreSQL scoring, migrations, monitoring, and automated tests live in the separate project scripts.

**Reproducibility:** Use the project's pinned requirements and Python version. Run cells from top to bottom. Cross-validation can take several minutes. The export cell writes `customer_churn_model.joblib` in the notebook's working directory; run it in a separate working copy if you want to preserve an existing deployed bundle. Changing package versions can change numerical results.

**Evaluation history:** An earlier exploratory loop accidentally included the original holdout. That loop is removed from this presentation, but its consequence is retained: the original holdout is not claimed as an untouched test set. Final evaluation instead used a new independent synthetic sample after model and threshold selection. Rerunning this notebook reproduces that fixed experiment; the sample is no longer a new unseen test for later tuning.


## 1. Generate synthetic customer data

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit

rng = np.random.default_rng(42)
n = 2000

tenure = rng.integers(1, 61, n)
monthly_fee = rng.uniform(150, 1200, n).round(2)
support_tickets = rng.poisson(1.5, n)
usage_hours = rng.gamma(shape=3, scale=8, size=n).round(1)
contract = rng.choice(
    ["Monthly", "Annual"], size=n, p=[0.7, 0.3]
)

# Synthetic outcome-generation rule—not a real business model.
log_odds = (
    -1.4
    - 0.025 * tenure
    + 0.001 * monthly_fee
    + 0.4 * support_tickets
    - 0.035 * usage_hours
    - 1.0 * (contract == "Annual")
)

churned = rng.binomial(1, expit(log_odds))

df = pd.DataFrame({
    "Customer_ID": np.arange(1, n + 1),
    "Tenure_Months": tenure,
    "Monthly_Fee": monthly_fee,
    "Support_Tickets": support_tickets,
    "Usage_Hours": usage_hours,
    "Contract_Type": contract,
    "Churned": churned
})

# Introduce missing features for preprocessing practice.
for column in ["Monthly_Fee", "Usage_Hours", "Contract_Type"]:
    missing = rng.choice(n, size=100, replace=False)
    df.loc[missing, column] = np.nan

print(df.head())
print("\nShape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

## 2. Reserve development data

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Customer_ID", "Churned"])
y = df["Churned"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Development rows:", len(X_dev))
print("Test rows:", len(X_test))
print("\nDevelopment target counts:")
print(y_dev.value_counts().sort_index())

## 3. Explore development data only

In [ ]:
dev = X_dev.copy()
dev["Churned"] = y_dev

print(dev.describe(include="all"))
print("\nMissing values:")
print(dev.isna().sum())

In [ ]:
dev_contract = dev.groupby(['Contract_Type'],dropna=False).agg(Customer_Total=('Churned','count'),Total_Churn= ('Churned','sum'),).reset_index()
dev_contract['Churn_Rate'] =100.0 * dev_contract['Total_Churn'] / dev_contract['Customer_Total']
dev_contract

In the synthetic development data, monthly contracts were associated with higher churn: 17.44%, compared with 9.89% for annual contracts. This association does not establish that contract type causes churn.

In [ ]:
dev_num = dev.select_dtypes(include='number')
dev_num.groupby('Churned').median()

## 4. Define preprocessing and candidate models

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
from sklearn.compose import ColumnTransformer

numeric_features = [
    "Tenure_Months",
    "Monthly_Fee",
    "Support_Tickets",
    "Usage_Hours"
]

categorical_features = ["Contract_Type"]

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [ ]:
from sklearn.linear_model import LogisticRegression

churn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
from sklearn.dummy import DummyClassifier

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

## 5. Baseline and default-threshold comparison

A missed churner costs R100 and a false positive costs R10 in this exercise. These are assumed error costs, not measured retention economics.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

baseline_predictions = cross_val_predict(
    baseline_pipeline,
    X_dev,
    y_dev,
    cv=cv,
    method="predict"
)

tn, fp, fn, tp = confusion_matrix(
    y_dev, baseline_predictions, labels=[0, 1]
).ravel()

total_cost = fn * 100 + fp * 10

print("Confusion matrix:")
print(np.array([[tn, fp], [fn, tp]]))
print("Total cost:", total_cost)
print("Cost per customer:", total_cost / len(y_dev))

In [ ]:
logistic_predictions = cross_val_predict(
    churn_pipeline,
    X_dev,
    y_dev,
    cv=cv,
    method="predict"
)

tn, fp, fn, tp = confusion_matrix(
    y_dev, logistic_predictions, labels=[0, 1]
).ravel()

total_cost = fn * 100 + fp * 10

print("Confusion matrix:")
print(np.array([[tn, fp], [fn, tp]]))
print("Total cost:", total_cost)
print("Cost per customer:", total_cost / len(y_dev))

## 6. Select thresholds using development predictions

In [ ]:
logistic_probabilities = cross_val_predict(
    churn_pipeline,
    X_dev,
    y_dev,
    cv=cv,
    method="predict_proba"
)[:, 1]
rows = []
for threshold in [0.05, 0.10, 0.15, 0.20, 0.25,0.30]:
    logistic_predictions = (logistic_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_dev, logistic_predictions, labels=[0, 1]
    ).ravel()

    total_cost = fn * 100 + fp * 10
    cost_per_customer = total_cost / len(y_dev)
    rows.append({'threshold':threshold,'FN':fn,'FP':fp,'Total_Cost':total_cost,'Cost_Per_Customer':cost_per_customer})
    dev_logistic = pd.DataFrame(rows).sort_values('Total_Cost')
dev_logistic

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

forest_probabilities = cross_val_predict(
    forest_pipeline,
    X_dev,
    y_dev,
    cv=cv,
    method="predict_proba"
)[:, 1]
rows = []
for threshold in [0.05, 0.10, 0.15, 0.20, 0.25,0.30]:
    forest_predictions = (forest_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_dev, forest_predictions, labels=[0, 1]
    ).ravel()

    total_cost = fn * 100 + fp * 10
    cost_per_customer = total_cost / len(y_dev)
    rows.append({'threshold':threshold,'FN':fn,'FP':fp,'Total_Cost':total_cost,'Cost_Per_Customer':cost_per_customer})
    dev_forest = pd.DataFrame(rows).sort_values('Total_Cost')
dev_forest

## 7. Nested cross-validation of model and threshold selection

Each outer validation fold evaluates the combination selected using inner out-of-fold predictions on outer training rows only. Ties favour logistic regression and then the lower threshold. The aggregate cost estimates the selection procedure, not a single fixed model.

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)


In [ ]:
from sklearn.base import clone

models = {
    "Logistic Regression": churn_pipeline,
    "Random Forest": forest_pipeline
}

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
results = []

for fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(X_dev, y_dev), 1
):
    X_outer_train = X_dev.iloc[train_idx]
    y_outer_train = y_dev.iloc[train_idx]

    X_outer_val = X_dev.iloc[val_idx]
    y_outer_val = y_dev.iloc[val_idx]

    best_cost = float("inf")
    best_model_name = None
    best_threshold = None

    # Select both the model and threshold internally.
    # Ties retain the first combination encountered:
    # logistic regression first, then lower thresholds.
    for name, model in models.items():
        inner_probabilities = cross_val_predict(
            clone(model),
            X_outer_train,
            y_outer_train,
            cv=inner_cv,
            method="predict_proba"
        )[:, 1]

        for threshold in thresholds:
            predictions = (
                inner_probabilities >= threshold
            ).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                y_outer_train,
                predictions,
                labels=[0, 1]
            ).ravel()

            cost = fn * 100 + fp * 10

            if cost < best_cost:
                best_cost = cost
                best_model_name = name
                best_threshold = threshold

    # Refit the internally selected winner.
    selected_model = clone(models[best_model_name])
    selected_model.fit(X_outer_train, y_outer_train)

    outer_probabilities = selected_model.predict_proba(
        X_outer_val
    )[:, 1]

    outer_predictions = (
        outer_probabilities >= best_threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_outer_val,
        outer_predictions,
        labels=[0, 1]
    ).ravel()

    results.append({
        "Fold": fold,
        "Selected_Model": best_model_name,
        "Selected_Threshold": best_threshold,
        "False_Negatives": fn,
        "False_Positives": fp,
        "Total_Cost": fn * 100 + fp * 10,
        "Validation_Size": len(y_outer_val)
    })

outer_results = pd.DataFrame(results)

print(outer_results)
print(
    "Overall cost per customer:",
    outer_results["Total_Cost"].sum()
    / outer_results["Validation_Size"].sum()
)

## 8. Freeze the final development-selected model

In [ ]:
from sklearn.base import clone

# Select on all development predictions, not by outer-fold win counts.
development_comparison = pd.concat([
    dev_logistic.assign(Model="Logistic Regression"),
    dev_forest.assign(Model="Random Forest"),
], ignore_index=True)
development_comparison["model_priority"] = (
    development_comparison["Model"].map({
        "Logistic Regression": 0,
        "Random Forest": 1,
    })
)
best = development_comparison.sort_values(
    ["Total_Cost", "model_priority", "threshold"]
).iloc[0]

final_threshold = float(best["threshold"])
final_model = clone(models[best["Model"]])
final_model.fit(X_dev, y_dev)
print("Selected model:", best["Model"])
print("Selected threshold:", final_threshold)


## 9. Evaluate on a fresh synthetic sample

In [ ]:
from scipy.special import expit

eval_rng = np.random.default_rng(2026)
n_eval = 400

tenure = eval_rng.integers(1, 61, n_eval)
monthly_fee = eval_rng.uniform(150, 1200, n_eval).round(2)
support_tickets = eval_rng.poisson(1.5, n_eval)
usage_hours = eval_rng.gamma(3, 8, n_eval).round(1)
contract = eval_rng.choice(
    ["Monthly", "Annual"], size=n_eval, p=[0.7, 0.3]
)

log_odds = (
    -1.4
    - 0.025 * tenure
    + 0.001 * monthly_fee
    + 0.4 * support_tickets
    - 0.035 * usage_hours
    - 1.0 * (contract == "Annual")
)

y_fresh = eval_rng.binomial(1, expit(log_odds))

X_fresh = pd.DataFrame({
    "Tenure_Months": tenure,
    "Monthly_Fee": monthly_fee,
    "Support_Tickets": support_tickets,
    "Usage_Hours": usage_hours,
    "Contract_Type": contract
})

# Introduce 5% missingness in the same three columns.
for column in ["Monthly_Fee", "Usage_Hours", "Contract_Type"]:
    missing_rows = eval_rng.choice(
        n_eval, size=20, replace=False
    )
    X_fresh.loc[missing_rows, column] = np.nan

In [ ]:
from sklearn.metrics import precision_score, recall_score

fresh_probabilities = final_model.predict_proba(X_fresh)[:, 1]
fresh_predictions = (
    fresh_probabilities >= final_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_fresh, fresh_predictions, labels=[0, 1]
).ravel()

fresh_cost = fn * 100 + fp * 10
baseline_cost = int(y_fresh.sum()) * 100

print("Confusion matrix:")
print(np.array([[tn, fp], [fn, tp]]))
print("Model cost:", fresh_cost)
print("Baseline cost:", baseline_cost)
print("Cost per customer:", fresh_cost / n_eval)
print("Precision:", precision_score(
    y_fresh, fresh_predictions, zero_division=0
))
print("Recall:", recall_score(
    y_fresh, fresh_predictions, zero_division=0
))

### Recorded project results

In the original working environment, development selection chose logistic regression at 0.10. Nested validation estimated R7.20 per customer. Fresh synthetic evaluation produced 144 true negatives, 193 false positives, 15 false negatives, and 48 true positives: 19.92% precision, 76.19% recall, and an illustrative total cost of R3,430 versus R6,300 for the no-churn baseline (45.56% lower).

The original holdout was accidentally used during development; it is not reported as an independent final test. The replacement evaluation sample follows the same synthetic generating process. These results do not establish real-world retention savings. Compare rerun outputs with these recorded results and investigate environment differences rather than adjusting the model to match the evaluation sample.


## 10. Export and verify the fitted bundle

In [ ]:
import joblib

model_bundle = {
    "pipeline": final_model,
    "threshold": final_threshold,
    "feature_columns": X_dev.columns.tolist(),
    "false_negative_cost": 100,
    "false_positive_cost": 10,
    "model_version": "churn_logistic_v1"
}

joblib.dump(model_bundle, "customer_churn_model.joblib")

In [ ]:
loaded_bundle = joblib.load("customer_churn_model.joblib")

loaded_probabilities = loaded_bundle["pipeline"].predict_proba(
    X_fresh
)[:, 1]

print(
    "Probabilities match:",
    np.allclose(fresh_probabilities, loaded_probabilities)
)

loaded_predictions = (
    loaded_probabilities >= loaded_bundle["threshold"]
).astype(int)

print(
    "Predictions match:",
    np.array_equal(fresh_predictions, loaded_predictions)
)

## 11. Score example customers

In [ ]:
new_customers = pd.DataFrame({
    "Tenure_Months": [3, 48],
    "Monthly_Fee": [950.0, 350.0],
    "Support_Tickets": [5, 0],
    "Usage_Hours": [8.0, 40.0],
    "Contract_Type": ["Monthly", "Annual"]
})

new_customers = new_customers[
    loaded_bundle["feature_columns"]
]

probabilities = loaded_bundle["pipeline"].predict_proba(
    new_customers
)[:, 1]

results = new_customers.copy()
results["Churn_Probability"] = probabilities
results["Flag_For_Review"] = (
    probabilities >= loaded_bundle["threshold"]
).astype(int)

results

## Pipeline handoff and limitations

The saved bundle is consumed by `score_churn.py`. That script owns PostgreSQL reads and writes, incremental selection, validation, run history, and monthly locking. The pytest files own validation and integration checks. Database connection examples, local account details, and obsolete notebook scoring implementations were removed from this notebook.

The example flags are decision-rule outputs, not guarantees of churn. Before operational use, align feature definitions and observation windows with training, evaluate on real data, validate calibration and error costs, and test prospective performance.

No live database connection or password is required to run this notebook. Only load trusted model bundles.
